# Lab Problem 4: R Project Implementation and Version Control Using GitHub

## Image Recognition & Classification with Keras in R

**Name:** Shravni Umrani  
**Roll No.:** 23102C0058  
**Course:** R Programming  
**Experiment:** Lab Problem 4  

---

### Aim
To implement an end-to-end image recognition and classification project in **R** using a **Convolutional Neural Network (CNN)** with Keras/TensorFlow, following the prescribed tutorial workflow, and to document the project for version control using Git and GitHub.

### Problem Statement
The project classifies images into two classes:

- **Fan**
- **Television**

The workflow follows the tutorial concept of using 12 images (6 fan images and 6 television images), preprocessing them, splitting them into training and testing data, building a CNN model, training the model, and checking its predictions.

> **Note:** This notebook is self-contained. It generates simple sample fan and television images programmatically so the complete workflow can be demonstrated without requiring separate image uploads.

## 1. Install and Load Required Packages

The main packages used are:

- `keras` – to build the Convolutional Neural Network
- `tensorflow` – backend used by Keras
- `png` – to read and write PNG images
- `grid` – for simple image display

In [ ]:
# Install packages only if they are not already available
packages <- c("keras", "tensorflow", "png")

for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(keras)
library(tensorflow)
library(png)

cat("Required R packages loaded successfully.\n")

## 2. Create the Dataset

The original tutorial uses **12 images**:

- 6 fan images
- 6 television images

To make this notebook portable and directly executable, 12 small sample images are generated using R itself. They are saved inside a local `dataset` folder.

In [ ]:
set.seed(123)

dir.create("dataset", showWarnings = FALSE)

img_size <- 100

# Function to create a simple fan-like RGB image
make_fan <- function(seed_value) {
  set.seed(seed_value)
  img <- array(1, dim = c(img_size, img_size, 3))

  cx <- 50 + sample(-5:5, 1)
  cy <- 50 + sample(-5:5, 1)

  # Light noisy background
  noise <- array(runif(img_size * img_size * 3, 0, 0.06),
                 dim = c(img_size, img_size, 3))
  img <- pmax(0, img - noise)

  # Central hub
  for (x in 1:img_size) {
    for (y in 1:img_size) {
      d <- sqrt((x-cx)^2 + (y-cy)^2)
      if (d < 9) {
        img[x,y,] <- c(0.15, 0.15, 0.15)
      }

      # Four blades
      blade1 <- abs(y-cy) < 5 && x > cx && x < cx+35
      blade2 <- abs(x-cx) < 5 && y > cy && y < cy+35
      blade3 <- abs(y-cy) < 5 && x < cx && x > cx-35
      blade4 <- abs(x-cx) < 5 && y < cy && y > cy-35

      if (blade1 || blade2 || blade3 || blade4) {
        img[x,y,] <- c(0.3, 0.45, 0.7)
      }
    }
  }
  img
}

# Function to create a simple television-like RGB image
make_tv <- function(seed_value) {
  set.seed(seed_value)
  img <- array(1, dim = c(img_size, img_size, 3))

  noise <- array(runif(img_size * img_size * 3, 0, 0.06),
                 dim = c(img_size, img_size, 3))
  img <- pmax(0, img - noise)

  x1 <- 20 + sample(-3:3, 1)
  x2 <- 80 + sample(-3:3, 1)
  y1 <- 25 + sample(-3:3, 1)
  y2 <- 70 + sample(-3:3, 1)

  # Television body
  img[x1:x2, y1:y2, ] <- 0.12

  # Screen
  img[(x1+5):(x2-5), (y1+5):(y2-5), 1] <- 0.2
  img[(x1+5):(x2-5), (y1+5):(y2-5), 2] <- 0.45
  img[(x1+5):(x2-5), (y1+5):(y2-5), 3] <- 0.75

  # Stand
  img[47:53, (y2+1):85, ] <- 0.15
  img[35:65, 84:88, ] <- 0.15

  img
}

# Generate 6 fan images
for (i in 1:6) {
  writePNG(make_fan(i), paste0("dataset/fan", i, ".png"))
}

# Generate 6 television images
for (i in 1:6) {
  writePNG(make_tv(100 + i), paste0("dataset/tv", i, ".png"))
}

cat("Dataset created successfully.\n")
cat("Total images:", length(list.files("dataset", pattern = "\\.png$")), "\n")
print(list.files("dataset"))

## 3. Read the Images

All generated images are read into R. Each image has a size of **100 × 100 × 3**, where 3 represents the RGB colour channels.

In [ ]:
image_files <- c(
  paste0("dataset/fan", 1:6, ".png"),
  paste0("dataset/tv", 1:6, ".png")
)

images <- lapply(image_files, readPNG)

cat("Number of images:", length(images), "\n")
cat("Dimension of first image:", dim(images[[1]]), "\n")

## 4. Display Sample Images

A sample fan image and television image are displayed to verify that the dataset has been created correctly.

In [ ]:
par(mfrow = c(1,2), mar = c(1,1,2,1))

plot.new()
rasterImage(as.raster(images[[1]]), 0, 0, 1, 1)
title("Sample Fan")

plot.new()
rasterImage(as.raster(images[[7]]), 0, 0, 1, 1)
title("Sample Television")

par(mfrow = c(1,1))

## 5. Split the Dataset into Training and Testing Data

Following the tutorial:

- First 4 fan images + first 4 television images → **Training data**
- Remaining 2 fan images + remaining 2 television images → **Testing data**

Therefore:

- Training images = 8
- Testing images = 4

In [ ]:
train_images <- images[c(1:4, 7:10)]
test_images  <- images[c(5:6, 11:12)]

cat("Training images:", length(train_images), "\n")
cat("Testing images :", length(test_images), "\n")

## 6. Resize Images to 32 × 32

CNN models train faster on smaller images. Therefore, the images are resized from **100 × 100** to **32 × 32**.

A small helper function performs nearest-neighbour resizing using base R indexing.

In [ ]:
resize_array <- function(img, new_h = 32, new_w = 32) {
  old_h <- dim(img)[1]
  old_w <- dim(img)[2]

  row_idx <- round(seq(1, old_h, length.out = new_h))
  col_idx <- round(seq(1, old_w, length.out = new_w))

  img[row_idx, col_idx, , drop = FALSE]
}

train_images_32 <- lapply(train_images, resize_array)
test_images_32  <- lapply(test_images, resize_array)

cat("New training image dimension:", dim(train_images_32[[1]]), "\n")
cat("New testing image dimension :", dim(test_images_32[[1]]), "\n")

## 7. Convert Image Lists into 4-Dimensional Arrays

Keras expects image data in the following form:

`number_of_images × height × width × channels`

Hence:

- Training array → `8 × 32 × 32 × 3`
- Testing array → `4 × 32 × 32 × 3`

In [ ]:
list_to_4d_array <- function(img_list) {
  n <- length(img_list)
  arr <- array(0, dim = c(n, 32, 32, 3))

  for (i in seq_len(n)) {
    arr[i,,,] <- img_list[[i]]
  }

  arr
}

x_train <- list_to_4d_array(train_images_32)
x_test  <- list_to_4d_array(test_images_32)

cat("Training data dimensions:", dim(x_train), "\n")
cat("Testing data dimensions :", dim(x_test), "\n")

## 8. Create Class Labels

The two classes are encoded as:

- `0` → Fan
- `1` → Television

Training labels contain four examples of each class and testing labels contain two examples of each class.

In [ ]:
y_train_raw <- c(rep(0, 4), rep(1, 4))
y_test_raw  <- c(rep(0, 2), rep(1, 2))

cat("Training labels:", y_train_raw, "\n")
cat("Testing labels :", y_test_raw, "\n")

## 9. Convert Labels to Categorical Format

For the CNN output layer, the labels are converted into one-hot categorical format.

In [ ]:
y_train <- to_categorical(y_train_raw, num_classes = 2)
y_test  <- to_categorical(y_test_raw, num_classes = 2)

print(y_train)
print(y_test)

## 10. Build the Convolutional Neural Network

The CNN contains:

1. Convolution layer
2. Max pooling layer
3. Second convolution layer
4. Second max pooling layer
5. Flatten layer
6. Dense hidden layer
7. Output layer with 2 classes

In [ ]:
model <- keras_model_sequential() |>
  layer_conv_2d(
    filters = 16,
    kernel_size = c(3,3),
    activation = "relu",
    input_shape = c(32,32,3)
  ) |>
  layer_max_pooling_2d(pool_size = c(2,2)) |>
  layer_conv_2d(
    filters = 32,
    kernel_size = c(3,3),
    activation = "relu"
  ) |>
  layer_max_pooling_2d(pool_size = c(2,2)) |>
  layer_flatten() |>
  layer_dense(units = 32, activation = "relu") |>
  layer_dense(units = 2, activation = "softmax")

summary(model)

## 11. Compile the CNN Model

The model uses:

- **Optimizer:** Adam
- **Loss:** Categorical Cross-Entropy
- **Metric:** Accuracy

In [ ]:
model |> compile(
  optimizer = "adam",
  loss = "categorical_crossentropy",
  metrics = c("accuracy")
)

cat("Model compiled successfully.\n")

## 12. Train the Model

The CNN is trained for 20 epochs. Since the dataset is intentionally small, a small batch size is used.

In [ ]:
set.seed(123)

history <- model |> fit(
  x = x_train,
  y = y_train,
  epochs = 20,
  batch_size = 2,
  verbose = 1
)

plot(history)

## 13. Evaluate the Model

The trained model is evaluated on the four testing images.

In [ ]:
score <- model |> evaluate(
  x_test,
  y_test,
  verbose = 0
)

cat("Test Loss    :", round(score["loss"], 4), "\n")
cat("Test Accuracy:", round(score["accuracy"] * 100, 2), "%\n")

## 14. Predict the Test Images

The network predicts the class probability for each testing image. The class having the highest probability is selected as the final prediction.

In [ ]:
probabilities <- model |> predict(x_test, verbose = 0)

predicted_class <- max.col(probabilities) - 1

class_name <- function(x) {
  ifelse(x == 0, "Fan", "Television")
}

results <- data.frame(
  Image = c("fan5.png", "fan6.png", "tv5.png", "tv6.png"),
  Actual = class_name(y_test_raw),
  Predicted = class_name(predicted_class),
  Fan_Probability = round(probabilities[,1], 4),
  Television_Probability = round(probabilities[,2], 4)
)

print(results)

## 15. Calculate Classification Accuracy Manually

The predicted classes are compared with the actual classes.

In [ ]:
manual_accuracy <- mean(predicted_class == y_test_raw) * 100

cat("Correct predictions:",
    sum(predicted_class == y_test_raw),
    "out of",
    length(y_test_raw),
    "\n")

cat("Classification Accuracy:",
    round(manual_accuracy, 2),
    "%\n")

## 16. Display Test Images with Predicted Classes

In [ ]:
par(mfrow = c(2,2), mar = c(1,1,3,1))

for (i in 1:4) {
  plot.new()
  rasterImage(as.raster(test_images[[i]]), 0, 0, 1, 1)
  title(
    paste0(
      "Actual: ", class_name(y_test_raw[i]),
      "\nPredicted: ", class_name(predicted_class[i])
    ),
    cex.main = 0.9
  )
}

par(mfrow = c(1,1))

# Result

An image recognition and classification project was implemented successfully in R using a Convolutional Neural Network.

The complete workflow included:

- Creating/collecting fan and television images
- Image preprocessing
- Resizing images
- Dividing images into training and testing data
- Converting images into Keras-compatible arrays
- Creating class labels
- Building a CNN
- Training the CNN
- Evaluating the model
- Predicting unseen test images

The experiment demonstrates how Keras and TensorFlow can be used through R for basic image classification.

# Conclusion

In this experiment, I implemented an end-to-end image classification project using R and a Convolutional Neural Network. The images were preprocessed and converted into a form suitable for the CNN. The model learned features from fan and television images and was then used to predict the classes of test images.

This experiment helped me understand practical image preprocessing, CNN architecture, model training, testing, prediction, and how an R data science project can be organized for GitHub version control.

# GitHub Version Control

The assignment also requires the project to be maintained using Git and GitHub.

### Suggested Repository Name

`R-Image-Classification-Keras`

### Suggested Files

```text
R-Image-Classification-Keras/
│
├── R_Programming_Assignment_4.ipynb
├── README.md
└── screenshots/
```

### Git Commands

Run these commands inside the project folder:

```bash
git init
git add R_Programming_Assignment_4.ipynb
git commit -m "Add initial R image classification notebook"

git add README.md
git commit -m "Add project documentation"

git add .
git commit -m "Add final results and project files"

git branch -M main
git remote add origin YOUR_GITHUB_REPOSITORY_URL
git push -u origin main
```

Using multiple meaningful commits satisfies the requirement of maintaining a proper Git commit history.

# README.md Content

Copy the following text into a file named `README.md` in the GitHub repository:

```markdown
# Image Recognition and Classification with Keras in R

## Objective
To implement an image recognition and classification project in R using a Convolutional Neural Network with Keras and TensorFlow.

## Problem
The model classifies images into two classes:
- Fan
- Television

## Dataset
The project follows a 12-image workflow consisting of 6 fan images and 6 television images. Eight images are used for training and four images are used for testing.

## R Packages Used
- keras
- tensorflow
- png

## Major Operations
1. Dataset preparation
2. Image preprocessing
3. Resizing images
4. Training/testing split
5. Array conversion
6. Label encoding
7. CNN model creation
8. Model training
9. Model evaluation
10. Image prediction

## How to Run
1. Open the notebook in an R-compatible Jupyter/Colab environment.
2. Run all cells in sequence.
3. Allow package installation if required.
4. Observe the generated dataset, model training, accuracy, and predictions.

## Result
The CNN successfully performs binary image classification between fan and television images.

## Author
Shravni Umrani  
Roll No.: 23102C0058
```

# Viva Questions

**1. What is CNN?**  
CNN stands for Convolutional Neural Network. It is a neural network commonly used for image recognition and classification.

**2. Why are images resized?**  
Images are resized so that all inputs have the same dimensions and model computation becomes faster.

**3. What are the classes in this project?**  
Fan and Television.

**4. How many images are used?**  
12 images: 6 fan images and 6 television images.

**5. How is the dataset divided?**  
8 images are used for training and 4 images are used for testing.

**6. What does a convolution layer do?**  
It extracts important visual features such as edges, shapes, and patterns.

**7. What does max pooling do?**  
It reduces the spatial size of feature maps while preserving important information.

**8. Why is softmax used?**  
Softmax converts the model output into probabilities for the two classes.

**9. Which optimizer is used?**  
Adam optimizer.

**10. Why is GitHub used?**  
GitHub stores the project, maintains version history, and helps document and share the implementation.